- 왼쪽 카메라 이미지 `*_left.png`
- 오른쪽 카메라 이미지 `*_right.png`
- 시차(disparity) 이미지 (`*_disp.png` 또는 `*_disp16.png`)
- confidnece 이미지 (`*_confidence.png` 등)

In [9]:
import os
import re
import cv2
import numpy as np


# -----------------------------
# 1) conf 파서 (필요한 값만)
# -----------------------------
def read_conf(conf_path: str, section: str = "LEFT_CAM_FHD"):
    """
    conf 파일에서 fx, fy, cx, cy, BaseLine만 읽는다.
    section은 이미지 해상도에 맞게 선택 (예: LEFT_CAM_FHD / LEFT_CAM_2K / LEFT_CAM_HD ...)
    """
    with open(conf_path, "r", encoding="utf-8") as f:
        txt = f.read()

    def get_value(sec, key):
        # [SECTION] ... key=123.45 형태 파싱
        pattern = rf"\[{re.escape(sec)}\](.*?)(?=\n\[|\Z)"
        m = re.search(pattern, txt, flags=re.S)
        if not m:
            raise ValueError(f"Section [{sec}] not found in {conf_path}")

        block = m.group(1)
        m2 = re.search(rf"{re.escape(key)}\s*=\s*([-+0-9.eE]+)", block)
        if not m2:
            raise ValueError(f"Key {key} not found in section [{sec}]")
        return float(m2.group(1))

    fx = get_value(section, "fx")
    fy = get_value(section, "fy")
    cx = get_value(section, "cx")
    cy = get_value(section, "cy")
    baseline_mm = get_value("STEREO", "BaseLine")  # 보통 mm

    return fx, fy, cx, cy, baseline_mm


# -----------------------------
# 2) disparity16 로드 + 스케일 보정
# -----------------------------
def load_disparity16(path: str, disp_scale: float = 16.0):
    """
    disp16.png가 16-bit grayscale이라고 가정.
    true_disp(px) = raw / disp_scale
    disp_scale은 데이터에 따라 1.0 또는 16.0 등으로 바뀔 수 있음.
    """
    raw = cv2.imread(path, cv2.IMREAD_UNCHANGED)  # uint16 예상
    if raw is None:
        raise FileNotFoundError(path)

    raw = raw.astype(np.float32)
    disp = raw / float(disp_scale)
    return disp, raw


def load_confidence_binary(path: str):
    """
    confidence.png가 0/255 또는 0/1 형태일 수 있으니
    >0 인 픽셀을 True로 처리.
    """
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    mask = img > 0
    return mask


# -----------------------------
# 3) disparity -> depth(m)
# -----------------------------
def disparity_to_depth_m(disp: np.ndarray, fx: float, baseline_mm: float, min_disp: float = 1.0):
    """
    depth(mm) = fx(px) * baseline(mm) / disparity(px)
    -> depth(m)로 반환
    """
    disp_safe = np.maximum(disp, min_disp)
    depth_mm = (fx * baseline_mm) / disp_safe
    depth_m = depth_mm / 1000.0
    return depth_m


# -----------------------------
# 4) depth -> 3D 점(ROI+conf 필터)
# -----------------------------
def backproject_points(depth_m: np.ndarray, conf_mask: np.ndarray,
                       fx: float, fy: float, cx: float, cy: float,
                       roi_y0_ratio: float = 0.6, sample_step: int = 2):
    """
    하단 ROI(기본: 아래 40%)에서만 점을 뽑고, conf_mask가 True인 픽셀만 사용.
    sample_step으로 다운샘플링해서 속도 개선.
    """
    H, W = depth_m.shape
    y0 = int(H * roi_y0_ratio)

    ys = np.arange(y0, H, sample_step)
    xs = np.arange(0, W, sample_step)
    grid_x, grid_y = np.meshgrid(xs, ys)
    u = grid_x.reshape(-1)
    v = grid_y.reshape(-1)

    valid = conf_mask[v, u] & np.isfinite(depth_m[v, u]) & (depth_m[v, u] > 0)
    u = u[valid]
    v = v[valid]
    Z = depth_m[v, u]

    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy

    pts = np.stack([X, Y, Z], axis=1)  # (N,3)
    return pts


# -----------------------------
# 5) 평면 피팅 (RANSAC)
# plane: n·x + d = 0
# -----------------------------
def fit_plane_from_3pts(p1, p2, p3):
    v1 = p2 - p1
    v2 = p3 - p1
    n = np.cross(v1, v2)
    norm = np.linalg.norm(n)
    if norm < 1e-9:
        return None
    n = n / norm
    d = -np.dot(n, p1)
    return n, d


def ransac_plane(points: np.ndarray, iters: int = 300, dist_thresh: float = 0.03, seed: int = 42):
    """
    points: (N,3) meters
    dist_thresh: 평면에서의 거리 임계값(미터). 0.03m=3cm 정도로 시작 추천.
    """
    rng = np.random.default_rng(seed)
    N = points.shape[0]
    if N < 200:
        raise ValueError(f"Not enough points for RANSAC: {N}")

    best_inliers = None
    best_model = None
    best_cnt = 0

    for _ in range(iters):
        idx = rng.choice(N, size=3, replace=False)
        p1, p2, p3 = points[idx]
        model = fit_plane_from_3pts(p1, p2, p3)
        if model is None:
            continue

        n, d = model
        # point-to-plane distance
        dist = np.abs(points @ n + d)  # since n is normalized
        inliers = dist < dist_thresh
        cnt = int(inliers.sum())

        if cnt > best_cnt:
            best_cnt = cnt
            best_inliers = inliers
            best_model = (n, d)

    if best_model is None:
        raise RuntimeError("RANSAC failed to find a plane")

    # inlier 점들로 최종 least squares refinement (SVD)
    inlier_pts = points[best_inliers]
    centroid = inlier_pts.mean(axis=0)
    X = inlier_pts - centroid
    _, _, vh = np.linalg.svd(X, full_matrices=False)
    n = vh[-1]  # smallest singular vector
    n = n / np.linalg.norm(n)
    d = -np.dot(n, centroid)

    return n, d, best_cnt / points.shape[0]


# -----------------------------
# 6) normal -> slope grade(%)
# -----------------------------
def slope_from_normal_grade(n: np.ndarray):
    """
    카메라 좌표: X right, Y down, Z forward 가정.
    전후 경사(grade)는 |n_z| / |n_y| 근사로 계산.
    """
    ny = float(n[1])
    nz = float(n[2])
    eps = 1e-6
    tan_theta = abs(nz) / max(abs(ny), eps)
    grade = tan_theta * 100.0
    return grade


# -----------------------------
# 7) 한 장 처리 파이프라인
# -----------------------------
def estimate_slope_from_files(conf_path, disp16_path, confbin_path,
                              cam_section="LEFT_CAM_FHD",
                              disp_scale=16.0,
                              roi_y0_ratio=0.6,
                              sample_step=2,
                              ransac_iters=300,
                              ransac_dist_thresh=0.03):
    fx, fy, cx, cy, baseline_mm = read_conf(conf_path, section=cam_section)

    disp, raw_disp = load_disparity16(disp16_path, disp_scale=disp_scale)
    conf_mask = load_confidence_binary(confbin_path)

    depth_m = disparity_to_depth_m(disp, fx=fx, baseline_mm=baseline_mm, min_disp=1.0)

    pts = backproject_points(
        depth_m, conf_mask, fx, fy, cx, cy,
        roi_y0_ratio=roi_y0_ratio, sample_step=sample_step
    )

    n, d, inlier_ratio = ransac_plane(pts, iters=ransac_iters, dist_thresh=ransac_dist_thresh)

    grade = slope_from_normal_grade(n)

    return {
        "grade_percent": grade,
        "normal": n,
        "inlier_ratio": inlier_ratio,
        "fx": fx, "fy": fy, "cx": cx, "cy": cy, "baseline_mm": baseline_mm,
        "disp_raw_minmax": (float(raw_disp.min()), float(raw_disp.max()))
    }

In [10]:
conf_path = "datas/Depth_001/Depth_001.conf"
disp16_path = "datas/Depth_001/ZED1_KSC_001032_disp16.png"
confbin_path = "datas/Depth_001/ZED1_KSC_001032_confidence.png"  # 또는 confidence_save 기반이면 다른 파일

result = estimate_slope_from_files(
    conf_path, disp16_path, confbin_path,
    cam_section="LEFT_CAM_FHD",   # 이미지 크기에 맞춰 바꾸기
    disp_scale=16.0,              # <-- 여기 중요! 필요하면 1.0으로 바꿔보기
    roi_y0_ratio=0.6,
    sample_step=2,
    ransac_iters=400,
    ransac_dist_thresh=0.03
)

print("Estimated grade(%):", result["grade_percent"])
print("Plane normal:", result["normal"])
print("Inlier ratio:", result["inlier_ratio"])
print("Raw disp16 min/max:", result["disp_raw_minmax"])

def grade_to_angle_deg(grade_percent):
    return np.degrees(np.arctan(grade_percent / 100.0))

grade = result["grade_percent"]
angle = grade_to_angle_deg(grade)

print(f"Grade: {grade:.2f}%")
print(f"Angle: {angle:.2f}°")

Estimated grade(%): 100000000.0
Plane normal: [0. 0. 1.]
Inlier ratio: 0.7258529329738774
Raw disp16 min/max: (0.0, 19609.0)
Grade: 100000000.00%
Angle: 90.00°
